# PackingSolver exploration

the visualisation bit needs plotly and also nbformat>=4.2.0

In [1]:
import json
import sys
from importlib.metadata import version
from pathlib import Path

import packingsolver.rectangle as psr

print("Python:", sys.version)
print("PackingSolver:", version("packingsolver"))

Python: 3.14.6 | packaged by Anaconda, Inc. | (main, Jun 18 2026, 21:18:42) [MSC v.1942 64 bit (AMD64)]
PackingSolver: 0.1.1340


Load one existing dataset

In [2]:
project_root = Path.cwd()

if not (project_root / "data").is_dir():
    project_root = project_root.parent

dataset_path = project_root / "data" / "synthetic" / "pilot_v1.json"

with dataset_path.open() as file:
    data = json.load(file)

datasets = data["datasets"]

In [3]:
dataset = next(
    dataset
    for dataset in datasets
    if dataset["dataset_id"]
    == "rect_v1_mixed_sheet2500x1250_n35_seed0"
)

parts = dataset["parts"]
sheet_w, sheet_h = dataset["sheet_size"]

print("Dataset:", dataset["dataset_id"])
print("Parts:", len(parts))
print("Sheet size:", (sheet_w, sheet_h))

Dataset: rect_v1_mixed_sheet2500x1250_n35_seed0
Parts: 35
Sheet size: (2500, 1250)


### Build the instance

- Using the plain bin-packing objective: place all requested parts while minimising the number of sheets. 
- PackingSolver distinguishes this from BinPackingWithLeftovers, which additionally considers the leftover on the last sheet. We should leave that extra objective out of the initial comparison.
- rotation not yet established

In [4]:
available_sheets = len(parts)

builder = psr.InstanceBuilder()
builder.set_objective(psr.Objective.BinPacking)

builder.add_bin_type(
    sheet_w,
    sheet_h,
    copies=available_sheets,
)

for part_id, width, height in parts:
    builder.add_item_type(
        width,
        height,
        copies=1,
    )

instance = builder.build()

### Run the solver

In [5]:
parameters = psr.OptimizeParameters()
parameters.time_limit = 5

output = psr.optimize(instance, parameters)

### Inspect and visualise the result

In [6]:
print(output)
print(output.solution)

Time (s):              5.01695
Is proven infeasible:  0
Primal bound:          5
Dual bound:            4
Gap:                   20%



In [9]:
psr.visualize(output.solution).show()

***
- Enable rotation
- The mapping preserves original part identities instead of relying on dimensions to identify parts. This matters when multiple parts have identical dimensions.

In [7]:
rotation = True
available_sheets = len(parts)

builder = psr.InstanceBuilder()
builder.set_objective(psr.Objective.BinPacking)

builder.add_bin_type(
    sheet_w,
    sheet_h,
    copies=available_sheets,
)

item_type_to_part_id = {}

for part_id, width, height in parts:
    item_type_id = builder.add_item_type(
        width,
        height,
        copies=1,
        oriented=not rotation,
    )

    item_type_to_part_id[item_type_id] = part_id

instance = builder.build()

Optimisiation cell

In [8]:
parameters = psr.OptimizeParameters()
parameters.time_limit = 5

output = psr.optimize(instance, parameters)

Export and inspect solution
- For ITEM rows, LX and LY are the dimensions after placement, so they already account for rotation. BIN identifies the sheet containing the item

In [9]:
import pandas as pd

results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

solution_path = results_dir / "pilot_solution.csv"

output.solution.write(str(solution_path))

solution_df = pd.read_csv(solution_path)

display(solution_df.head(10))
print("Columns:", solution_df.columns.tolist())

,TYPE,ID,COPIES,BIN,X,Y,LX,LY,GROUP_ID
0,BIN,0,1,0,0,0,2500,1250,NaN
1,ITEM,28,1,0,0,0,1307,500,0.0
2,ITEM,18,1,0,1307,0,601,1240,0.0
3,ITEM,29,1,0,0,500,1268,341,0.0
4,ITEM,2,1,0,1908,0,588,821,0.0
5,ITEM,21,1,0,0,841,1306,308,0.0
6,ITEM,12,1,0,1908,821,508,391,0.0
7,BIN,0,1,1,0,0,2500,1250,NaN
8,ITEM,15,1,1,0,0,1362,327,0.0
9,ITEM,1,1,1,1362,0,597,1247,0.0


Columns: ['TYPE', 'ID', 'COPIES', 'BIN', 'X', 'Y', 'LX', 'LY', 'GROUP_ID']


Convert to shared layout format

This produces precisely the representation the RectPack adapter returns: (bin_id, x, y, width, height, part_id)

In [10]:
bin_rows = (
    solution_df.loc[solution_df["TYPE"] == "BIN"]
    .sort_values("BIN")
)

item_rows = solution_df.loc[solution_df["TYPE"] == "ITEM"]

if bin_rows.empty or item_rows.empty:
    raise ValueError(
        "No non-empty packing was exported; inspect the solver output."
    )

if not bin_rows["COPIES"].eq(1).all():
    raise ValueError(
        "This initial converter expects one physical sheet per BIN row."
    )

bin_id_map = {
    int(original_bin_id): new_bin_id
    for new_bin_id, original_bin_id in enumerate(bin_rows["BIN"])
}

used_sheets = [
    (int(row.LX), int(row.LY))
    for row in bin_rows.itertuples(index=False)
]

placements = [
    (
        bin_id_map[int(row.BIN)],
        int(row.X),
        int(row.Y),
        int(row.LX),
        int(row.LY),
        item_type_to_part_id[int(row.ID)],
    )
    for row in item_rows.itertuples(index=False)
]

print("Used sheets:", len(used_sheets))
print("Placed parts:", len(placements))
print("First placements:", placements[:5])

Used sheets: 5
Placed parts: 35
First placements: [(0, 0, 0, 1307, 500, 'P00028'), (0, 1307, 0, 601, 1240, 'P00018'), (0, 0, 500, 1268, 341, 'P00029'), (0, 1908, 0, 588, 821, 'P00002'), (0, 0, 841, 1306, 308, 'P00021')]


Run shared metrics

In [11]:
from cutting_benchmark.rectangle_metrics import (
    calculate_rectangle_metrics,
)

metrics = calculate_rectangle_metrics(
    parts=parts,
    placements=placements,
    used_sheets=used_sheets,
    sheet_size=(sheet_w, sheet_h),
    rotation=rotation,
)

display(pd.Series(metrics, name="PackingSolver"))

parts_total                      35
parts_packed                     35
complete                       True
individually_feasible          True
sheets_used                       5
requested_area             12133094
packed_area                12133094
part_fraction                   1.0
area_fraction                   1.0
utilisation                0.776518
area_lower_bound                  4
sheet_gap_to_area_bound           1
Name: PackingSolver, dtype: object

check identity preservation explicitly:

In [12]:
requested_ids = {part_id for part_id, _, _ in parts}
placed_ids = [placement[5] for placement in placements]

assert len(placed_ids) == len(set(placed_ids)), "Duplicate part placements"
assert set(placed_ids) <= requested_ids, "Unknown part IDs"

missing_ids = requested_ids - set(placed_ids)

print("Missing parts:", sorted(missing_ids))

Missing parts: []


Layout validation

In [14]:
from cutting_benchmark.rectangle_validation import validate_rectangle_layout

validation = validate_rectangle_layout(
    parts=parts,
    placements=placements,
    used_sheets=used_sheets,
    sheet_size=(sheet_w, sheet_h),
    rotation=rotation,
    available_sheets=available_sheets,
)

if not validation["layout_valid"]:
    raise ValueError(
        "Invalid PackingSolver layout:\n"
        + "\n".join(validation["validation_errors"])
    )

metrics = calculate_rectangle_metrics(
    parts=parts,
    placements=placements,
    used_sheets=used_sheets,
    sheet_size=(sheet_w, sheet_h),
    rotation=rotation,
)

display(pd.Series({
    "layout_valid": validation["layout_valid"],
    **metrics,
}))

layout_valid                   True
parts_total                      35
parts_packed                     35
complete                       True
individually_feasible          True
sheets_used                       5
requested_area             12133094
packed_area                12133094
part_fraction                   1.0
area_fraction                   1.0
utilisation                0.776518
area_lower_bound                  4
sheet_gap_to_area_bound           1
dtype: object

*** 
Try the packing solver function from the cutting_benchmark package

In [15]:
from cutting_benchmark.solvers.packingsolver_adapter import (
    run_pack as run_packingsolver,
)

result = run_packingsolver(
    parts=dataset["parts"],
    sheet_size=dataset["sheet_size"],
    rotation=True,
    time_limit=5,
    return_layout=True,
)

display(pd.Series({
    key: value
    for key, value in result.items()
    if key not in {"placements", "used_sheets"}
}))

solver                          packingsolver
config_id                  BinPacking__time5s
objective                          BinPacking
rotation                                 True
time_limit_seconds                          5
sheet_width                              2500
sheet_height                             1250
available_sheets                           35
runtime_seconds                      5.015091
layout_valid                             True
parts_total                                35
parts_packed                               35
complete                                 True
individually_feasible                    True
sheets_used                                 5
requested_area                       12133094
packed_area                          12133094
part_fraction                             1.0
area_fraction                             1.0
utilisation                          0.776518
area_lower_bound                            4
sheet_gap_to_area_bound           

In [16]:
display(
    pd.DataFrame(
        result["placements"],
        columns=[
            "bin_id",
            "x",
            "y",
            "width",
            "height",
            "part_id",
        ],
    ).head()
)

,bin_id,x,y,width,height,part_id
0,0,0,0,1061,141,P00000
1,0,0,141,1247,597,P00001
2,0,0,738,777,243,P00004
3,0,0,981,835,201,P00006
4,0,777,738,1111,193,P00022


Run a small benchmark

In [17]:
from cutting_benchmark.benchmark import run_benchmark

configurations = [
    {"rotation": False, "time_limit": 5},
    {"rotation": True, "time_limit": 5},
]

benchmark_results = run_benchmark(
    datasets=[dataset],
    configurations=configurations,
    solver=run_packingsolver,
    repeats=1,
)

In [20]:
display(benchmark_results)

,solver,config_id,objective,rotation,time_limit_seconds,sheet_width,sheet_height,available_sheets,runtime_seconds,layout_valid,...,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound,run_status,dataset_id,seed,profile,n_parts,repeat
0,packingsolver,BinPacking__time5s,BinPacking,True,5,2500,1250,35,5.014526,True,...,1.0,0.776518,4,1,solution_returned,rect_v1_mixed_sheet2500x1250_n35_seed0,0,mixed,35,0
1,packingsolver,BinPacking__time5s,BinPacking,False,5,2500,1250,35,5.022970,True,...,1.0,0.776518,4,1,solution_returned,rect_v1_mixed_sheet2500x1250_n35_seed0,0,mixed,35,0


In [18]:
results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

benchmark_results.to_csv(
    results_dir / "packingsolver_pilot_raw.csv",
    index=False,
)

***
## Try on multiple datasets

In [21]:
with dataset_path.open() as file:
    data = json.load(file)

datasets = data["datasets"]

In [22]:
from cutting_benchmark.benchmark import run_benchmark

configurations = [
    {"rotation": False, "time_limit": 5},
    {"rotation": True, "time_limit": 5},
]

In [23]:
n_runs = len(datasets) * len(configurations)

print("Datasets:", len(datasets))
print("Configurations:", len(configurations))
print("Total runs:", n_runs)

Datasets: 30
Configurations: 2
Total runs: 60


With 5 seconds time limit it can take up to 5 minutes to run the following cell. 

In [54]:
from cutting_benchmark.solvers.packingsolver_adapter import (
    run_pack as run_packingsolver,
)

benchmark_results_5s = run_benchmark(
    datasets=datasets,
    configurations=configurations,
    solver=run_packingsolver,
    repeats=1,
)

benchmark_results_5s.head()

,solver,config_id,objective,rotation,time_limit_seconds,sheet_width,sheet_height,available_sheets,runtime_seconds,run_status,...,part_fraction,area_fraction,utilisation,area_lower_bound,sheet_gap_to_area_bound,dataset_id,seed,profile,n_parts,repeat
0,packingsolver,BinPacking__time5s,BinPacking,True,5,2500,1250,100,9.346769,no_solution_returned,...,0.0,0.0,0.000000,20,NaN,rect_v1_large_and_small_sheet2500x1250_n100_seed1,1,large_and_small,100,0
1,packingsolver,BinPacking__time5s,BinPacking,False,5,2500,1250,35,0.268224,solution_returned,...,1.0,1.0,0.799891,3,0.0,rect_v1_strips_sheet2500x1250_n35_seed3,3,strips,35,0
2,packingsolver,BinPacking__time5s,BinPacking,True,5,2500,1250,100,9.927442,no_solution_returned,...,0.0,0.0,0.000000,10,NaN,rect_v1_mixed_sheet2500x1250_n100_seed3,3,mixed,100,0
3,packingsolver,BinPacking__time5s,BinPacking,True,5,2500,1250,35,0.253202,solution_returned,...,1.0,1.0,0.797313,3,0.0,rect_v1_strips_sheet2500x1250_n35_seed0,0,strips,35,0
4,packingsolver,BinPacking__time5s,BinPacking,False,5,2500,1250,100,5.124975,solution_returned,...,1.0,1.0,0.844404,7,1.0,rect_v1_strips_sheet2500x1250_n100_seed1,1,strips,100,0


In [55]:
results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

benchmark_path = results_dir / "benchmark_results_5s.csv"
benchmark_results_5s.to_csv(benchmark_path, index=False)

print("Benchmark results exported to:", benchmark_path)
print("Runs:", len(benchmark_results))
print(
    "Runs with no solution returned:",
    benchmark_results["run_status"].eq("no_solution_returned").sum(),
)

display(
    benchmark_results[
        [
            "dataset_id",
            "config_id",
            "run_status",
            "parts_total",
            "parts_packed",
            "complete",
        ]
    ]
)

Benchmark results exported to: c:\Users\ivezs\2D-Optimisation-in-Metals-Manufacturing\results\packingsolver\benchmark_results_5s.csv
Runs: 2
Runs with no solution returned: 1


,dataset_id,config_id,run_status,parts_total,parts_packed,complete
0,rect_v1_large_and_small_sheet2500x1250_n100_seed1,BinPacking__time5s,no_solution_returned,100,0,False
1,rect_v1_large_and_small_sheet2500x1250_n100_seed1,BinPacking__time5s,solution_returned,100,100,True


In [30]:
import pandas as pd

# run_benchmark records one row per dataset/configuration, including runs
# where PackingSolver returned no populated solution.
expected_runs = len(configurations)
expected_dataset_ids = pd.Index(
    [dataset["dataset_id"] for dataset in datasets],
    name="dataset_id",
)

summary = benchmark_results.groupby("dataset_id").agg(
    runs_recorded=("run_status", "size"),
    solutions_returned=("run_status", lambda s: s.eq("solution_returned").sum()),
    no_solution_returned=("run_status", lambda s: s.eq("no_solution_returned").sum()),
    complete_layouts=("complete", "sum"),
)
summary = summary.reindex(expected_dataset_ids)
summary["runs_recorded"] = summary["runs_recorded"].fillna(0).astype(int)
summary["runs_expected"] = expected_runs
summary["runs_missing"] = summary["runs_expected"] - summary["runs_recorded"]
summary[["solutions_returned", "no_solution_returned", "complete_layouts"]] = (
    summary[["solutions_returned", "no_solution_returned", "complete_layouts"]]
    .fillna(0)
    .astype(int)
)

print("PackingSolver time limit: 5 seconds per run")
print(f"Datasets requested: {len(expected_dataset_ids)}")
print(f"Datasets with no solution returned in any configuration: {(summary['solutions_returned'] == 0).sum()}")
print(f"Datasets with at least one complete layout: {(summary['complete_layouts'] > 0).sum()}")
display(summary)

print("\nResults by configuration:")
display(
    benchmark_results.groupby(["config_id", "run_status"], dropna=False)
    .size()
    .unstack(fill_value=0)
)

PackingSolver time limit: 5 seconds per run
Datasets requested: 30
Datasets with no solution returned in any configuration: 0
Datasets with at least one complete layout: 30


,runs_recorded,solutions_returned,no_solution_returned,complete_layouts,runs_expected,runs_missing
dataset_id,,,,,,
rect_v1_mixed_sheet2500x1250_n35_seed0,2,2,0,2,2,0
rect_v1_mixed_sheet2500x1250_n35_seed1,2,2,0,2,2,0
rect_v1_mixed_sheet2500x1250_n35_seed2,2,2,0,2,2,0
rect_v1_mixed_sheet2500x1250_n35_seed3,2,2,0,2,2,0
rect_v1_mixed_sheet2500x1250_n35_seed4,2,2,0,2,2,0
rect_v1_strips_sheet2500x1250_n35_seed0,2,2,0,2,2,0
rect_v1_strips_sheet2500x1250_n35_seed1,2,2,0,2,2,0
rect_v1_strips_sheet2500x1250_n35_seed2,2,2,0,2,2,0
rect_v1_strips_sheet2500x1250_n35_seed3,2,2,0,2,2,0



Results by configuration:


run_status,no_solution_returned,solution_returned
config_id,,
BinPacking__time5s,8,52


### Check why there is no solution:

Check whether any requested parts cannot fit individually:

A useful check, without running the solver, is whether some datasets contain parts that cannot fit a sheet under a particular rotation setting.

In [31]:
fit_issues = []

for current_dataset in datasets:
    sheet_w, sheet_h = current_dataset["sheet_size"]

    for configuration in configurations:
        rotation = configuration["rotation"]

        unfit_ids = [
            part_id
            for part_id, width, height in current_dataset["parts"]
            if not (
                (width <= sheet_w and height <= sheet_h)
                or (
                    rotation
                    and height <= sheet_w
                    and width <= sheet_h
                )
            )
        ]

        if unfit_ids:
            fit_issues.append({
                "dataset_id": current_dataset["dataset_id"],
                "rotation": rotation,
                "unfit_count": len(unfit_ids),
                "unfit_part_ids": unfit_ids,
            })

display(pd.DataFrame(fit_issues))

""


There are no fit issues.

Lets investigate a single failing dataset:

rect_v1_large_and_small_sheet2500x1250_n100_seed1

In [32]:
failing_dataset = next(
    dataset
    for dataset in datasets
    if dataset["dataset_id"]
    == "rect_v1_large_and_small_sheet2500x1250_n100_seed1"
)

parts = failing_dataset["parts"]
sheet_w, sheet_h = failing_dataset["sheet_size"]
rotation = True
available_sheets = len(parts)

In [38]:
display(
    benchmark_results.loc[
        (benchmark_results["dataset_id"] == failing_dataset["dataset_id"])
        & benchmark_results["rotation"],
        ["dataset_id", "config_id", "run_status", "parts_total", "parts_packed", "complete", "runtime_seconds"],
    ]
)

,dataset_id,config_id,run_status,parts_total,parts_packed,complete,runtime_seconds
0,rect_v1_large_and_small_sheet2500x1250_n100_seed1,BinPacking__time5s,no_solution_returned,100,0,False,9.191774


The solver packed 0 parts out of 100. 

In [35]:
results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

debug_path = results_dir / "failing_solution.csv"

output.solution.write(str(debug_path))

debug_df = pd.read_csv(debug_path)

print("Certificate shape:", debug_df.shape)
display(debug_df.head(20))

Certificate shape: (0, 9)


,TYPE,ID,COPIES,BIN,X,Y,LX,LY,GROUP_ID


Let's try to run the solver with a bigger time limit: 
before 5s
now 30s

In [36]:
parameters.time_limit = 30

output_longer = psr.optimize(instance, parameters)

print(output_longer)

Time (s):              11.0221
Is proven infeasible:  0
Primal bound:          38
Dual bound:            38
Gap:                   0%



With longer time the solver found a solution and placed all items. 

In [16]:
debug_longer_path = results_dir / "failing_solution_30s.csv"

output_longer.solution.write(str(debug_longer_path))

debug_longer_df = pd.read_csv(debug_longer_path)

print("Certificate shape:", debug_longer_df.shape)
display(debug_longer_df.head(20))

Certificate shape: (138, 9)


,TYPE,ID,COPIES,BIN,X,Y,LX,LY,GROUP_ID
0,BIN,0,1,0,0,0,2500,1250,NaN
1,ITEM,1,1,0,0,0,1386,623,0.0
2,ITEM,7,1,0,0,623,1730,615,0.0
3,BIN,0,1,1,0,0,2500,1250,NaN
4,ITEM,50,1,1,0,0,1400,619,0.0
5,ITEM,59,1,1,0,619,1389,618,0.0
6,BIN,0,1,2,0,0,2500,1250,NaN
7,ITEM,88,1,2,0,0,1982,757,0.0
8,BIN,0,1,3,0,0,2500,1250,NaN
9,ITEM,77,1,3,0,0,1972,962,0.0


Conclusion: No solution was returned during the shorter time run, longer run returned all the 100 parts 

In [17]:
result = run_packingsolver(
    parts=failing_dataset["parts"],
    sheet_size=failing_dataset["sheet_size"],
    rotation=True,
    time_limit=30,
    return_layout=True,
)

display(pd.Series({
    key: value
    for key, value in result.items()
    if key not in {"placements", "used_sheets"}
}))

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  100
Number of items:       100
Number of bin types:   1
Number of bins:        100
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       62016631
Mean item width:       930.13
Mean item height:      462.45
Smallest item width:   63
Smallest item height:  63
Total item profit:     6.20166e+07
Largest item profit:   1.89706e+06
Total item weight:     0
Largest item copies:   1
Total bin area:        312500000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
      10.078      38           47.78                                

Final statistics
----------------
Time (s):              10.078
Is proven infeasible:  0
Primal bound:          38
Dual bound:

solver                           packingsolver
config_id                  BinPacking__time30s
objective                           BinPacking
rotation                                  True
time_limit_seconds                          30
sheet_width                               2500
sheet_height                              1250
available_sheets                           100
runtime_seconds                      10.078943
layout_valid                              True
parts_total                                100
parts_packed                               100
complete                                  True
individually_feasible                     True
sheets_used                                 38
requested_area                        62016631
packed_area                           62016631
part_fraction                              1.0
area_fraction                              1.0
utilisation                           0.522245
area_lower_bound                            20
sheet_gap_to_

Rerun the benchmark with 30s time limit

In [18]:
configurations = [
    {"rotation": False, "time_limit": 30},
    {"rotation": True, "time_limit": 30},
]

benchmark_results = run_benchmark(
    datasets=datasets,
    configurations=configurations,
    solver=run_packingsolver,
    repeats=1,
)

          PackingSolver          

Problem type
------------
Rectangle

Instance
--------
Objective:             BinPacking
Number of item types:  100
Number of items:       100
Number of bin types:   1
Number of bins:        100
Number of groups:      1
Number of defects:     0
Unloading constraint:  None
Total item area:       62016631
Mean item width:       930.13
Mean item height:      462.45
Smallest item width:   63
Smallest item height:  63
Total item profit:     6.20166e+07
Largest item profit:   1.89706e+06
Total item weight:     0
Largest item copies:   1
Total bin area:        312500000
Total bin weight:      inf
Largest bin cost:      3.125e+06

        Time    Bins  Full waste (%)                         Comment
        ----    ----  --------------                         -------
      10.687      38           47.78                                

Final statistics
----------------
Time (s):              10.6874
Is proven infeasible:  0
Primal bound:          38
Dual bound

In [19]:
results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

benchmark_results.to_csv(
    results_dir / "packingsolver_pilot_raw.csv",
    index=False,
)

In [ ]:
# this needs to be ran again after restarting the kernel with updated benchmark.py

display(
    benchmark_results.groupby(
        ["run_status", "rotation"]
    )
    .size()
    .rename("runs")
    .reset_index()
)

KeyError: 'run_status'

In [22]:
display(
    benchmark_results.groupby(
        ["rotation", "complete", "parts_packed"]
    )
    .size()
    .rename("runs")
    .reset_index()
)

,rotation,complete,parts_packed,runs
0,False,True,35,15
1,False,True,100,15
2,True,True,35,15
3,True,True,100,15


In [23]:
summary = (
    benchmark_results
    .groupby("rotation")
    .agg(
        runs=("dataset_id", "size"),
        complete_runs=("complete", "sum"),
        empty_runs=("parts_packed", lambda values: (values == 0).sum()),
        partial_runs=(
            "parts_packed",
            lambda values: (
                (values > 0)
                & (values < benchmark_results.loc[values.index, "parts_total"])
            ).sum()
        ),
        mean_runtime_seconds=("runtime_seconds", "mean"),
        max_runtime_seconds=("runtime_seconds", "max"),
    )
    .reset_index()
)

display(summary)

,rotation,runs,complete_runs,empty_runs,partial_runs,mean_runtime_seconds,max_runtime_seconds
0,False,30,30,0,0,10.684481,32.694300
1,True,30,30,0,0,11.508370,30.369061


In [24]:
successful_results = benchmark_results.loc[
    benchmark_results["complete"]
]

display(
    successful_results[
        [
            "dataset_id",
            "rotation",
            "parts_packed",
            "parts_total",
            "sheets_used",
            "utilisation",
            "runtime_seconds",
        ]
    ]
)

,dataset_id,rotation,parts_packed,parts_total,sheets_used,utilisation,runtime_seconds
0,rect_v1_large_and_small_sheet2500x1250_n100_seed1,True,100,100,38,0.522245,10.688380
1,rect_v1_strips_sheet2500x1250_n35_seed3,False,35,35,3,0.799891,0.597895
2,rect_v1_mixed_sheet2500x1250_n100_seed3,True,100,100,11,0.884692,30.229841
3,rect_v1_strips_sheet2500x1250_n35_seed0,True,35,35,3,0.797313,0.497239
4,rect_v1_strips_sheet2500x1250_n100_seed1,False,100,100,8,0.844404,31.916142
5,rect_v1_mixed_sheet2500x1250_n100_seed4,True,100,100,11,0.882372,30.178932
6,rect_v1_large_and_small_sheet2500x1250_n35_seed1,True,35,35,16,0.457779,0.386401
7,rect_v1_strips_sheet2500x1250_n35_seed2,False,35,35,3,0.768552,0.333186
8,rect_v1_large_and_small_sheet2500x1250_n100_seed2,True,100,100,39,0.548993,7.340294
9,rect_v1_large_and_small_sheet2500x1250_n100_seed2,False,100,100,42,0.509779,0.774353


In [ ]:
results_dir = project_root / "results" / "packingsolver"
results_dir.mkdir(parents=True, exist_ok=True)

successful_results.to_csv(
    results_dir / "packingsolver_benchmark_raw.csv",
    index=False,
)